# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

The dataset contains protein abundance and modification data from human extracellular vesicles, described using the Croissant schema standard.

For reproducibility, all dataset entities (record sets, fields, columns) are referenced by their `@id`.

### Dataset Source
The Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata (access attributes, not subscripts)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

A Croissant dataset organizes data into **record sets** (tables). Each record set has fields (columns), identified by their unique `@id`.

Let's enumerate the record sets, fields, and columns.

In [ ]:
# List available record sets and their @id
record_sets = dataset.metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', 'No name')} (description: {rs.get('description', 'No description')})")
    print("  Fields/columns in this record set:")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']} : {field.get('name', 'No name')} (type: {field.get('dataType', 'N/A')})")
    print()

In [ ]:
# For illustration, print a few rows from each record set using their @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Sample records from record set {rs_id}:")
    count = 0
    for rec in dataset.records(record_set=rs_id):
        print(rec)
        count += 1
        if count == 2:
            break
    print()

## 3. Data Extraction

Here we load data for each record set into pandas DataFrames for further analysis. All Croissant entity references use their `@id` values.

**Step guidance:**
- Choose a record set of interest (e.g., likely to contain the main protein abundance table, often named with `protein`, `abundance`, etc).
- You can list all record set IDs in the `record_sets_ids` list.

In [ ]:
# Gather all record set @ids
record_sets_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        # Show a preview
        display(dataframes[record_set_id].head())

# If you want to work on a specific record set:
# Example: first record set
main_record_set_id = record_sets_ids[0] if record_sets_ids else None

## 4. Exploratory Data Analysis (EDA)
Here, we demonstrate basic filtering, normalization, and grouping using `@id` of fields, as required by the schema.

- Pick a numeric field (e.g., 'abundance', 'MW', etc.)
- Filter records by a cutoff
- Normalize a numeric field
- Group results by a categorical field (e.g., experimental condition if provided)


In [ ]:
# Example: Pick the main record set and fields for analysis
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Inspect available columns and their @id
    print(f"Available columns in {main_record_set_id}:")
    print(df.columns.tolist())

    # Try to select a numeric field (try likely names)
    # Change this to the actual @id from above for your analysis
    possible_numeric = [col for col in df.columns if any(x in col.lower() for x in ['abundance', 'mw', 'coverage', 'peptide', 'intensity', 'quant', 'score'])]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
            print(filtered_df.head())

            # Normalize
            norm_field = f"{numeric_field_id}_normalized"
            filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, norm_field]].head())

            # Try group by a likely categorical field
            cat_fields = [col for col in df.columns if any(x in col.lower() for x in ['type', 'condition', 'class', 'group', 'sample'])]
            if cat_fields:
                group_field = cat_fields[0]
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
                print(f"Mean {numeric_field_id} grouped by {group_field}:")
                print(grouped_df.head())
    else:
        print("No obvious numeric field found in columns.")

## 5. Visualization

Let's plot the numeric field distribution and group means to visualize protein abundance (or any other selected numeric field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the numeric field's distribution
if main_record_set_id and possible_numeric:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If we found a group field, plot mean per group
    if cat_fields:
        fig, ax = plt.subplots(figsize=(10, 4))
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
        group_means.plot(kind='bar', ax=ax)
        ax.set_ylabel(f"Mean {numeric_field_id}")
        ax.set_title(f"Mean {numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 protein dataset using `mlcroissant`, referenced all entities by their Croissant `@id`, and demonstrated basic exploratory data analysis and visualization. This approach ensures your exploration is reproducible, transparent, and compatible with the FAIR data ecosystem.

- For robust research, always reference fields/tables by `@id`.
- Extend this notebook for deeper analytics, such as clustering, statistical testing, or integration with annotation metadata.

See the [original FAIR^2 dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for more metadata and context.